# Rice Disease Dataset — Research Audit

## Purpose

This notebook documents the audit and preprocessing decisions for the rice
source-domain dataset used in the cross-crop transfer-learning study.

The audit follows the same methodology used for the Pearl Millet target dataset
so that dataset quality, class distribution, image integrity, duplication,
metadata consistency, and preprocessing decisions can be reported consistently
in the research paper.

## Research Role

- Source domain: Rice
- Target domain: Pearl Millet
- Primary purpose: Cross-crop knowledge transfer
- Dataset use: Source-domain training / representation learning
- Audit principle: Preserve the original dataset state before any cleaning

## Audit Stages

1. Dataset provenance and inventory
2. Metadata / CSV consistency
3. Class distribution
4. Image integrity and readability
5. Image properties and resolution
6. Exact duplicate detection
7. Near-duplicate detection
8. Visual inspection
9. Data-quality observations
10. Final cleaned dataset statistics
11. Research decisions and exclusions

No images are deleted during the initial audit.
All modifications, if required, are performed only after the original
dataset state has been documented.

## The rice dataset we used is Paddy Doctor: Paddy Disease Classification, hosted on Kaggle.
Official dataset link:

Paddy Doctor — Download Dataset on KaggleThe dataset provides 10,407 labeled rice-leaf images across 10 classes, along with train.csv containing image labels and metadata. This matches the dataset we audited in your project. 

For your research documentation, use this citation:
Paddy Doctor, Pandarasamy Arjunan, and Petchiammal. Paddy Doctor: Paddy Disease Classification. Kaggle, 2022. https://www.kaggle.com/competitions/paddy-disease-classification. 


Important: Kaggle lists the data as subject to its competition rules. Review those terms before redistributing the dataset or publishing it with your research. 



In [5]:
from pathlib import Path

# ============================================================
# Rice Dataset Audit — Paths
# ============================================================

# Project root
PROJECT_DIR = Path.home() / "cate_model" / "Pearl_millet_Prediction"

# Rice dataset
RICE_DIR = PROJECT_DIR / "Rice_Dataset"

# Dataset directories
RAW_DIR = RICE_DIR / "Raw"
EXTRACTED_DIR = RICE_DIR / "Extracted"
IMAGE_DIR = EXTRACTED_DIR / "train_images"
AUDIT_DIR = RICE_DIR / "Audit"

# Metadata
CSV_PATH = EXTRACTED_DIR / "train.csv"

# Create audit directory
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

print("Project directory:")
print(PROJECT_DIR)

print("\nRice dataset:")
print(RICE_DIR)

print("\nCSV:")
print(CSV_PATH)

print("\nCSV exists:", CSV_PATH.exists())
print("Image directory exists:", IMAGE_DIR.exists())

Project directory:
/home/zahidrashid/cate_model/Pearl_millet_Prediction

Rice dataset:
/home/zahidrashid/cate_model/Pearl_millet_Prediction/Rice_Dataset

CSV:
/home/zahidrashid/cate_model/Pearl_millet_Prediction/Rice_Dataset/Extracted/train.csv

CSV exists: True
Image directory exists: True


Step 3: Load train.csv
## 1. Metadata Inventory

The dataset metadata file (`train.csv`) is inspected before any image
modification.

The metadata contains:

- `image_id` — image filename
- `label` — rice disease / healthy class
- `variety` — rice variety
- `age` — plant age

The metadata is treated as the reference source for verifying whether the
extracted image collection is complete and correctly organized.

In [6]:
df = pd.read_csv(CSV_PATH)

print("Number of metadata records:", len(df))
print("\nColumns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nFirst 5 records:")
display(df.head())

Number of metadata records: 10407

Columns:
['image_id', 'label', 'variety', 'age']

Data types:
image_id    object
label       object
variety     object
age          int64
dtype: object

First 5 records:


,image_id,label,variety,age
0,100330.jpg,bacterial_leaf_blight,ADT45,45
1,100365.jpg,bacterial_leaf_blight,ADT45,45
2,100382.jpg,bacterial_leaf_blight,ADT45,45
3,100632.jpg,bacterial_leaf_blight,ADT45,45
4,101918.jpg,bacterial_leaf_blight,ADT45,45


Next: Metadata quality audit
## 2. Metadata Quality Audit

This section checks the integrity of the metadata before examining or modifying
the image files.

The following checks are performed:

- Missing values
- Duplicate metadata rows
- Duplicate image IDs
- Number of unique disease/healthy labels
- Number of rice varieties
- Plant-age distribution

These checks establish whether the metadata itself contains inconsistencies
that could affect subsequent dataset analysis or model training.

In [7]:
# ============================================================
# Metadata Quality Audit
# ============================================================

print("=== Missing Values ===")
display(df.isnull().sum())

print("\n=== Duplicate Metadata Rows ===")
print(df.duplicated().sum())

print("\n=== Duplicate Image IDs ===")
print(df["image_id"].duplicated().sum())

print("\n=== Unique Image IDs ===")
print(df["image_id"].nunique())

print("\n=== Unique Labels ===")
print(df["label"].nunique())

print("\n=== Labels ===")
print(sorted(df["label"].unique()))

print("\n=== Unique Rice Varieties ===")
print(df["variety"].nunique())

print("\n=== Rice Varieties ===")
print(sorted(df["variety"].unique()))

print("\n=== Plant Age Statistics ===")
display(df["age"].describe())

=== Missing Values ===


image_id    0
label       0
variety     0
age         0
dtype: int64


=== Duplicate Metadata Rows ===
0

=== Duplicate Image IDs ===
0

=== Unique Image IDs ===
10407

=== Unique Labels ===
10

=== Labels ===
['bacterial_leaf_blight', 'bacterial_leaf_streak', 'bacterial_panicle_blight', 'blast', 'brown_spot', 'dead_heart', 'downy_mildew', 'hispa', 'normal', 'tungro']

=== Unique Rice Varieties ===
10

=== Rice Varieties ===
['ADT45', 'AndraPonni', 'AtchayaPonni', 'IR20', 'KarnatakaPonni', 'Onthanel', 'Ponni', 'RR', 'Surya', 'Zonal']

=== Plant Age Statistics ===


count    10407.000000
mean        64.043624
std          8.958830
min         45.000000
25%         60.000000
50%         67.000000
75%         70.000000
max         82.000000
Name: age, dtype: float64

In [8]:
# Save metadata summary for the audit record

metadata_summary = pd.DataFrame({
    "metric": [
        "total_records",
        "unique_image_ids",
        "duplicate_image_ids",
        "duplicate_rows",
        "unique_labels",
        "unique_varieties",
        "missing_values"
    ],
    "value": [
        len(df),
        df["image_id"].nunique(),
        df["image_id"].duplicated().sum(),
        df.duplicated().sum(),
        df["label"].nunique(),
        df["variety"].nunique(),
        int(df.isnull().sum().sum())
    ]
})

metadata_summary.to_csv(
    AUDIT_DIR / "rice_metadata_summary.csv",
    index=False
)

display(metadata_summary)

,metric,value
0,total_records,10407
1,unique_image_ids,10407
2,duplicate_image_ids,0
3,duplicate_rows,0
4,unique_labels,10
5,unique_varieties,10
6,missing_values,0


## 3. Class Distribution and Image-Directory Consistency

The class distribution reported in `train.csv` is compared with the number of
images physically available in the extracted image directories.

This comparison is performed before any cleaning or duplicate removal.

The purpose is to identify:

- Missing image files
- Unexpected extra files
- Class-level extraction discrepancies
- Differences between metadata labels and directory organization

Any discrepancy is documented before further dataset preprocessing.

In [9]:
# ============================================================
# CSV Class Distribution
# ============================================================

csv_class_counts = (
    df["label"]
    .value_counts()
    .sort_index()
)

print("Class distribution according to train.csv:\n")
print(csv_class_counts)

print("\nTotal records:", csv_class_counts.sum())

Class distribution according to train.csv:

label
bacterial_leaf_blight        479
bacterial_leaf_streak        380
bacterial_panicle_blight     337
blast                       1738
brown_spot                   965
dead_heart                  1442
downy_mildew                 620
hispa                       1594
normal                      1764
tungro                      1088
Name: count, dtype: int64

Total records: 10407


In [10]:
# ============================================================
# Actual Image-Folder Distribution
# ============================================================

image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

folder_counts = {}

for class_dir in sorted(IMAGE_DIR.iterdir()):
    if class_dir.is_dir():
        count = sum(
            1
            for f in class_dir.iterdir()
            if f.is_file() and f.suffix.lower() in image_extensions
        )
        folder_counts[class_dir.name] = count

folder_class_counts = pd.Series(folder_counts).sort_index()

print("Actual image-folder distribution:\n")
print(folder_class_counts)

print("\nTotal images found:", folder_class_counts.sum())

Actual image-folder distribution:

bacterial_leaf_blight        479
bacterial_leaf_streak        380
bacterial_panicle_blight     337
blast                       1738
brown_spot                   965
dead_heart                  1442
downy_mildew                 620
hispa                       1594
normal                         0
tungro                       614
dtype: int64

Total images found: 8169


In [11]:
# ============================================================
# CSV vs Image Folder Comparison
# ============================================================

comparison = pd.DataFrame({
    "csv_count": csv_class_counts,
    "folder_count": folder_class_counts
}).fillna(0).astype(int)

comparison["difference"] = (
    comparison["csv_count"] - comparison["folder_count"]
)

comparison["status"] = comparison["difference"].apply(
    lambda x: "MATCH" if x == 0 else "MISMATCH"
)

display(comparison)

,csv_count,folder_count,difference,status
bacterial_leaf_blight,479,479,0,MATCH
bacterial_leaf_streak,380,380,0,MATCH
bacterial_panicle_blight,337,337,0,MATCH
blast,1738,1738,0,MATCH
brown_spot,965,965,0,MATCH
dead_heart,1442,1442,0,MATCH
downy_mildew,620,620,0,MATCH
hispa,1594,1594,0,MATCH
normal,1764,0,1764,MISMATCH
tungro,1088,614,474,MISMATCH


## Initial Class-Consistency Finding

The metadata contains 10,407 unique image records, while the currently
extracted image directories contain 8,169 image files.

Eight of the ten classes have an exact match between the metadata and the
extracted image directories.

Two classes show discrepancies:

- `normal`: 1,764 records are listed in the metadata, but no corresponding
  images are currently present in the extracted image directory.
- `tungro`: 1,088 records are listed in the metadata, while 614 images are
  currently present.

This results in a total discrepancy of 2,238 image records.

At this stage, the cause of the discrepancy has not been established.
Therefore, no images are deleted, relabeled, or excluded based on this
finding. The original metadata and extracted dataset state are preserved
while the missing records are investigated.

In [12]:
# ============================================================
# Identify Metadata Records Missing from Extracted Folders
# ============================================================

# Collect all image filenames currently available
actual_image_names = set()

for class_dir in IMAGE_DIR.iterdir():
    if class_dir.is_dir():
        for f in class_dir.iterdir():
            if f.is_file() and f.suffix.lower() in image_extensions:
                actual_image_names.add(f.name)

# Image IDs expected according to CSV
csv_image_names = set(df["image_id"].astype(str))

# Missing and unexpected files
missing_image_ids = sorted(csv_image_names - actual_image_names)
extra_image_ids = sorted(actual_image_names - csv_image_names)

print("Images listed in CSV:", len(csv_image_names))
print("Images found in folders:", len(actual_image_names))
print("Missing from folders:", len(missing_image_ids))
print("Extra images in folders:", len(extra_image_ids))

# Metadata for missing images
missing_df = df[
    df["image_id"].astype(str).isin(missing_image_ids)
].copy()

print("\n=== Missing records by class ===")
print(missing_df["label"].value_counts())

print("\n=== Missing records by variety ===")
print(missing_df["variety"].value_counts())

print("\n=== Missing records by age ===")
print(missing_df["age"].value_counts().sort_index())

# Save complete missing-record audit
missing_df.to_csv(
    AUDIT_DIR / "rice_missing_image_records.csv",
    index=False
)

print("\nSaved audit file:")
print(AUDIT_DIR / "rice_missing_image_records.csv")

Images listed in CSV: 10407
Images found in folders: 8169
Missing from folders: 2238
Extra images in folders: 0

=== Missing records by class ===
label
normal    1764
tungro     474
Name: count, dtype: int64

=== Missing records by variety ===
variety
ADT45           1884
Ponni            291
AtchayaPonni      21
RR                18
Onthanel          12
Zonal             10
Surya              2
Name: count, dtype: int64

=== Missing records by age ===
age
45      49
50     197
55      55
57      72
60    1050
65      64
67     114
70     496
72      72
73      10
75      14
77      10
80      30
82       5
Name: count, dtype: int64

Saved audit file:
/home/zahidrashid/cate_model/Pearl_millet_Prediction/Rice_Dataset/Audit/rice_missing_image_records.csv


## Missing-Image Metadata Profile

The 2,238 records absent from the extracted image directory belong exclusively
to the `normal` and `tungro` classes.

The missing records are associated with multiple rice varieties and plant ages,
rather than a single metadata category. The largest missing representation is
associated with the ADT45 variety.

Because no extra image files were found outside the metadata, the discrepancy
cannot be attributed to unrelated or incorrectly named image files in the
current extracted directory.

The cause of the missing images is still unresolved and must be investigated
against the original `train_images.zip` before the dataset is considered
complete.

In [13]:
from zipfile import ZipFile

ZIP_PATH = RAW_DIR / "train_images.zip"

print("ZIP exists:", ZIP_PATH.exists())
print("ZIP size (MB):", round(ZIP_PATH.stat().st_size / (1024**2), 2))

with ZipFile(ZIP_PATH, "r") as z:
    members = z.namelist()

print("Total ZIP entries:", len(members))

# Show examples of the archive structure
print("\nFirst 20 entries:")
for item in members[:20]:
    print(item)

ZIP exists: True
ZIP size (MB): 781.7
Total ZIP entries: 10407

First 20 entries:
bacterial_leaf_blight/100023.jpg
bacterial_leaf_blight/100049.jpg
bacterial_leaf_blight/100126.jpg
bacterial_leaf_blight/100133.jpg
bacterial_leaf_blight/100148.jpg
bacterial_leaf_blight/100162.jpg
bacterial_leaf_blight/100169.jpg
bacterial_leaf_blight/100234.jpg
bacterial_leaf_blight/100248.jpg
bacterial_leaf_blight/100268.jpg
bacterial_leaf_blight/100289.jpg
bacterial_leaf_blight/100330.jpg
bacterial_leaf_blight/100365.jpg
bacterial_leaf_blight/100382.jpg
bacterial_leaf_blight/100445.jpg
bacterial_leaf_blight/100447.jpg
bacterial_leaf_blight/100513.jpg
bacterial_leaf_blight/100516.jpg
bacterial_leaf_blight/100523.jpg
bacterial_leaf_blight/100541.jpg


In [14]:
# ============================================================
# Inspect ZIP Contents by Class
# ============================================================

zip_class_counts = {}

for member in members:
    parts = Path(member).parts

    # Expected structure:
    # train_images/class_name/image.jpg
    if len(parts) >= 3:
        class_name = parts[-2]
        filename = parts[-1]

        if Path(filename).suffix.lower() in image_extensions:
            zip_class_counts[class_name] = (
                zip_class_counts.get(class_name, 0) + 1
            )

print("Image counts inside ZIP:\n")

for class_name, count in sorted(zip_class_counts.items()):
    print(f"{class_name}: {count}")

print("\nTotal images inside ZIP:", sum(zip_class_counts.values()))

Image counts inside ZIP:


Total images inside ZIP: 0


In [15]:
#Add this corrected cell
# ============================================================
# Inspect ZIP Contents by Class — Corrected
# ============================================================

zip_class_counts = {}

for member in members:
    path = Path(member)

    # ZIP structure:
    # class_name/image.jpg
    if len(path.parts) >= 2:
        class_name = path.parts[-2]
        filename = path.parts[-1]

        if Path(filename).suffix.lower() in image_extensions:
            zip_class_counts[class_name] = (
                zip_class_counts.get(class_name, 0) + 1
            )

print("Image counts inside ZIP:\n")

for class_name, count in sorted(zip_class_counts.items()):
    print(f"{class_name}: {count}")

print("\nTotal images inside ZIP:", sum(zip_class_counts.values()))

Image counts inside ZIP:

bacterial_leaf_blight: 479
bacterial_leaf_streak: 380
bacterial_panicle_blight: 337
blast: 1738
brown_spot: 965
dead_heart: 1442
downy_mildew: 620
hispa: 1594
normal: 1764
tungro: 1088

Total images inside ZIP: 10407


In [16]:
# And add this verification cell
# ============================================================
# Compare ZIP vs CSV
# ============================================================

zip_image_names = set()

for member in members:
    path = Path(member)

    if len(path.parts) >= 2:
        filename = path.parts[-1]

        if Path(filename).suffix.lower() in image_extensions:
            zip_image_names.add(filename)

csv_image_names = set(df["image_id"].astype(str))

missing_from_zip = sorted(csv_image_names - zip_image_names)
extra_in_zip = sorted(zip_image_names - csv_image_names)

print("CSV image IDs:", len(csv_image_names))
print("ZIP image files:", len(zip_image_names))
print("CSV records missing from ZIP:", len(missing_from_zip))
print("Extra ZIP images not in CSV:", len(extra_in_zip))

CSV image IDs: 10407
ZIP image files: 10407
CSV records missing from ZIP: 0
Extra ZIP images not in CSV: 0


# Phase 2.4 — Recover and verify the complete rice dataset

In [ ]:
# Step 1 — Add a Markdown cell to your notebook



## Phase 2.4: Dataset Extraction Recovery and Verification

### Objective

Recover the complete image dataset from the original Paddy Doctor ZIP archive and verify its consistency with the accompanying CSV metadata.

### Initial Finding

The original ZIP archive contains 10,407 image files, matching all 10,407 unique image IDs in `train.csv`. No missing or extra image filenames were found when comparing the archive against the CSV.

However, the previously extracted working directory contained only 8,169 images. The missing 2,238 images were associated with the `normal` and `tungro` classes.

### Recovery Procedure

The original ZIP archive will be used as the authoritative source. Missing files will be extracted into their corresponding class folders, preserving existing files.

### Verification Criteria

1. The extracted image count must equal 10,407.
2. Every CSV image ID must exist in the extracted dataset.
3. No unexpected image files should exist.
4. The extracted class distribution must match the CSV class distribution.

No duplicate removal or image modification will be performed during this recovery phase.


In [3]:
#path inspection
from pathlib import Path

# Project directories
PROJECT_DIR = (
    Path.home()
    / "cate_model"
    / "Pearl_millet_Prediction"
)

RICE_DIR = PROJECT_DIR / "Rice_Dataset"

# Raw source files
RAW_DIR = RICE_DIR / "Raw"
ZIP_PATH = RAW_DIR / "train_images.zip"

# Extracted dataset and metadata
EXTRACTED_DIR = RICE_DIR / "Extracted"
IMAGE_DIR = EXTRACTED_DIR / "train_images"
CSV_PATH = EXTRACTED_DIR / "train.csv"

# Audit reports
AUDIT_DIR = RICE_DIR / "Audit"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

print("Raw directory:", RAW_DIR)
print("ZIP exists:", ZIP_PATH.exists())
print("Extracted directory:", EXTRACTED_DIR)
print("Image directory exists:", IMAGE_DIR.exists())
print("CSV exists:", CSV_PATH.exists())
print("Audit directory:", AUDIT_DIR)

Raw directory: /home/zahidrashid/cate_model/Pearl_millet_Prediction/Rice_Dataset/Raw
ZIP exists: True
Extracted directory: /home/zahidrashid/cate_model/Pearl_millet_Prediction/Rice_Dataset/Extracted
Image directory exists: True
CSV exists: True
Audit directory: /home/zahidrashid/cate_model/Pearl_millet_Prediction/Rice_Dataset/Audit


In [4]:
# ============================================================
# Phase 2.4 — Step 1: Inspect Before Recovery
# ============================================================

from zipfile import ZipFile

image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

print("ZIP exists:", ZIP_PATH.exists())
print("CSV exists:", CSV_PATH.exists())
print("Image directory exists:", IMAGE_DIR.exists())

existing_images = [
    p for p in IMAGE_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in image_extensions
]

print("Images currently extracted:", len(existing_images))
print("ZIP size (MB):", round(ZIP_PATH.stat().st_size / (1024 ** 2), 2))

with ZipFile(ZIP_PATH, "r") as archive:
    archive_image_count = sum(
        1 for member in archive.infolist()
        if not member.is_dir()
        and Path(member.filename).suffix.lower() in image_extensions
    )

print("Images inside original ZIP:", archive_image_count)

ZIP exists: True
CSV exists: True
Image directory exists: True
Images currently extracted: 8169
ZIP size (MB): 781.7
Images inside original ZIP: 10407


In [5]:
# ============================================================
# Phase 2.4 — Step 2: Recover Missing Images
# ============================================================

import shutil
from zipfile import ZipFile

recovered = 0
already_present = 0
skipped_entries = 0

with ZipFile(ZIP_PATH, "r") as archive:
    for member in archive.infolist():

        if member.is_dir():
            continue

        relative_path = Path(member.filename)

        # Expected ZIP structure: class_name/image.jpg
        if len(relative_path.parts) != 2:
            skipped_entries += 1
            continue

        class_name = relative_path.parts[0]
        filename = relative_path.name

        if relative_path.suffix.lower() not in image_extensions:
            skipped_entries += 1
            continue

        destination = IMAGE_DIR / class_name / filename

        if destination.exists():
            already_present += 1
            continue

        destination.parent.mkdir(parents=True, exist_ok=True)

        with archive.open(member) as source:
            with open(destination, "wb") as target:
                shutil.copyfileobj(source, target)

        recovered += 1

print("Recovery completed.")
print("New images recovered:", recovered)
print("Existing images preserved:", already_present)
print("Other entries skipped:", skipped_entries)

Recovery completed.
New images recovered: 2238
Existing images preserved: 8169
Other entries skipped: 0


In [6]:
# ============================================================
# Phase 2.4 — Step 3: Verify Extraction Against CSV
# ============================================================

import pandas as pd
from collections import Counter

df = pd.read_csv(CSV_PATH)

image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

# Load all extracted image paths
extracted_images = [
    p for p in IMAGE_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in image_extensions
]

# Compare filenames against CSV
csv_image_names = set(df["image_id"].astype(str))
extracted_image_names = {p.name for p in extracted_images}

missing_from_extracted = sorted(
    csv_image_names - extracted_image_names
)
extra_in_extracted = sorted(
    extracted_image_names - csv_image_names
)

# Count images by class folder
folder_class_counts = Counter(
    p.parent.name for p in extracted_images
)

csv_class_counts = df["label"].value_counts().to_dict()

# Report
print("=" * 65)
print("RICE DATASET — EXTRACTION VERIFICATION")
print("=" * 65)

print("CSV records:", len(df))
print("Unique CSV image IDs:", len(csv_image_names))
print("Extracted images:", len(extracted_images))
print("Missing images:", len(missing_from_extracted))
print("Extra images:", len(extra_in_extracted))

print("\nCLASS DISTRIBUTION")
print("-" * 65)

for class_name in sorted(csv_class_counts):
    expected = csv_class_counts[class_name]
    actual = folder_class_counts.get(class_name, 0)
    status = "MATCH" if expected == actual else "MISMATCH"

    print(
        f"{class_name:30s} "
        f"CSV={expected:5d} | "
        f"Extracted={actual:5d} | {status}"
    )

# Overall verification
passed = (
    len(extracted_images) == len(csv_image_names)
    and not missing_from_extracted
    and not extra_in_extracted
    and all(
        csv_class_counts[class_name]
        == folder_class_counts.get(class_name, 0)
        for class_name in csv_class_counts
    )
)

print("\n" + "=" * 65)
print("FINAL RESULT:", "PASS" if passed else "CHECK REQUIRED")
print("=" * 65)

RICE DATASET — EXTRACTION VERIFICATION
CSV records: 10407
Unique CSV image IDs: 10407
Extracted images: 10407
Missing images: 0
Extra images: 0

CLASS DISTRIBUTION
-----------------------------------------------------------------
bacterial_leaf_blight          CSV=  479 | Extracted=  479 | MATCH
bacterial_leaf_streak          CSV=  380 | Extracted=  380 | MATCH
bacterial_panicle_blight       CSV=  337 | Extracted=  337 | MATCH
blast                          CSV= 1738 | Extracted= 1738 | MATCH
brown_spot                     CSV=  965 | Extracted=  965 | MATCH
dead_heart                     CSV= 1442 | Extracted= 1442 | MATCH
downy_mildew                   CSV=  620 | Extracted=  620 | MATCH
hispa                          CSV= 1594 | Extracted= 1594 | MATCH
normal                         CSV= 1764 | Extracted= 1764 | MATCH
tungro                         CSV= 1088 | Extracted= 1088 | MATCH

FINAL RESULT: PASS


## Phase 2.4: Dataset Extraction Recovery and Verification — Completed

### Results

The original Paddy Doctor archive contained 10,407 images, while the initial extracted working directory contained 8,169 images. A total of 2,238 missing images were recovered from the original archive without overwriting existing files.

The extracted dataset was subsequently compared against `train.csv`.

| Verification check                    | Result |
| ------------------------------------- | -----: |
| CSV records                           | 10,407 |
| Unique CSV image IDs                  | 10,407 |
| Extracted images                      | 10,407 |
| Images missing from extracted dataset |      0 |
| Extra images not listed in CSV        |      0 |
| Class distributions matching CSV      |  10/10 |

### Conclusion

The extraction verification passed. Every CSV image ID was found in the extracted dataset, no unexpected image filenames were detected, and the image count for every class matched the metadata.

The dataset is ready for image-integrity and visual-quality auditing. This verification confirms file inventory and metadata consistency; it does not establish that every image is readable, unique, or correctly labelled.


In [7]:
# ============================================================
# Phase 2.5 — Image Integrity, Dimensions and Color Modes
# ============================================================

from PIL import Image, UnidentifiedImageError
from collections import Counter, defaultdict
import pandas as pd

image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

image_records = []
unreadable_images = []

for image_path in sorted(IMAGE_DIR.rglob("*")):
    if not image_path.is_file():
        continue

    if image_path.suffix.lower() not in image_extensions:
        continue

    try:
        with Image.open(image_path) as img:
            img.verify()

        # Reopen after verify() to inspect actual image properties
        with Image.open(image_path) as img:
            img.load()

            image_records.append({
                "image_id": image_path.name,
                "class_name": image_path.parent.name,
                "relative_path": str(image_path.relative_to(IMAGE_DIR)),
                "format": img.format,
                "width": img.width,
                "height": img.height,
                "mode": img.mode,
                "file_size_bytes": image_path.stat().st_size,
                "readable": True,
            })

    except (UnidentifiedImageError, OSError, ValueError) as exc:
        unreadable_images.append({
            "relative_path": str(image_path.relative_to(IMAGE_DIR)),
            "error": str(exc),
        })

image_df = pd.DataFrame(image_records)

print("=" * 65)
print("RICE DATASET — IMAGE INTEGRITY AUDIT")
print("=" * 65)
print("Total image files found:", len(image_records) + len(unreadable_images))
print("Successfully decoded:", len(image_records))
print("Unreadable/corrupted:", len(unreadable_images))

if not image_df.empty:
    print("\nImage formats:")
    print(image_df["format"].value_counts(dropna=False).to_string())

    print("\nColor modes:")
    print(image_df["mode"].value_counts(dropna=False).to_string())

    print("\nImage dimensions:")
    print("Unique width-height combinations:",
          image_df[["width", "height"]].drop_duplicates().shape[0])
    print("Minimum width:", image_df["width"].min())
    print("Maximum width:", image_df["width"].max())
    print("Minimum height:", image_df["height"].min())
    print("Maximum height:", image_df["height"].max())

    print("\nFile sizes (MB):")
    print("Minimum:", round(image_df["file_size_bytes"].min() / (1024 ** 2), 4))
    print("Maximum:", round(image_df["file_size_bytes"].max() / (1024 ** 2), 4))
    print("Total:", round(image_df["file_size_bytes"].sum() / (1024 ** 2), 2))

    print("\nReadable images by class:")
    print(image_df.groupby("class_name").size().sort_index().to_string())

# Save audit records without changing the source images
image_df.to_csv(AUDIT_DIR / "rice_image_integrity.csv", index=False)
pd.DataFrame(unreadable_images).to_csv(
    AUDIT_DIR / "rice_unreadable_images.csv", index=False
)

print("\nAudit reports saved to:", AUDIT_DIR)

RICE DATASET — IMAGE INTEGRITY AUDIT
Total image files found: 10407
Successfully decoded: 10407
Unreadable/corrupted: 0

Image formats:
format
JPEG    10407

Color modes:
mode
RGB    10407

Image dimensions:
Unique width-height combinations: 2
Minimum width: 480
Maximum width: 640
Minimum height: 480
Maximum height: 640

File sizes (MB):
Minimum: 0.0298
Maximum: 0.1164
Total: 781.55

Readable images by class:
class_name
bacterial_leaf_blight        479
bacterial_leaf_streak        380
bacterial_panicle_blight     337
blast                       1738
brown_spot                   965
dead_heart                  1442
downy_mildew                 620
hispa                       1594
normal                      1764
tungro                      1088

Audit reports saved to: /home/zahidrashid/cate_model/Pearl_millet_Prediction/Rice_Dataset/Audit


## Phase 2.5: Image Integrity and Properties — Completed

All 10,407 extracted rice images were successfully decoded. No unreadable or corrupted images were detected.

All images were JPEG files using RGB color mode. The dataset contained two unique width-height combinations, with widths ranging from 480 to 640 pixels and heights ranging from 480 to 640 pixels. The combined image file size was approximately 781.55 MB.

The integrity audit records were saved as `rice_image_integrity.csv`, and the unreadable-image report was saved as `rice_unreadable_images.csv`.

### Conclusion

The dataset passed the basic image-integrity checks. This result confirms that the files can be decoded; it does not establish image uniqueness, label correctness, or independence of physical specimens.


In [8]:
# ============================================================
# Phase 2.6 — Resolution and Image-Property Distribution
# ============================================================

import pandas as pd

INTEGRITY_CSV = AUDIT_DIR / "rice_image_integrity.csv"
image_df = pd.read_csv(INTEGRITY_CSV)

print("=" * 70)
print("RICE DATASET — RESOLUTION AUDIT")
print("=" * 70)

print("\n1. Exact image dimensions:")
dimension_counts = (
    image_df.groupby(["width", "height"])
    .size()
    .reset_index(name="image_count")
    .sort_values("image_count", ascending=False)
)

print(dimension_counts.to_string(index=False))

print("\n2. Dimensions by class:")
class_resolution = (
    image_df.groupby(["class_name", "width", "height"])
    .size()
    .reset_index(name="image_count")
    .sort_values(["class_name", "width", "height"])
)

print(class_resolution.to_string(index=False))

print("\n3. Image aspect ratios:")
image_df["aspect_ratio"] = (
    image_df["width"] / image_df["height"]
).round(4)

print(image_df["aspect_ratio"].value_counts().sort_index().to_string())

print("\n4. Image dimensions summary:")
print(
    image_df[["width", "height"]]
    .describe()
    .round(2)
    .to_string()
)

dimension_counts.to_csv(
    AUDIT_DIR / "rice_dimension_distribution.csv",
    index=False
)

class_resolution.to_csv(
    AUDIT_DIR / "rice_class_resolution_distribution.csv",
    index=False
)

print("\nReports saved:")
print(AUDIT_DIR / "rice_dimension_distribution.csv")
print(AUDIT_DIR / "rice_class_resolution_distribution.csv")

RICE DATASET — RESOLUTION AUDIT

1. Exact image dimensions:
 width  height  image_count
   480     640        10403
   640     480            4

2. Dimensions by class:
              class_name  width  height  image_count
   bacterial_leaf_blight    480     640          477
   bacterial_leaf_blight    640     480            2
   bacterial_leaf_streak    480     640          380
bacterial_panicle_blight    480     640          337
                   blast    480     640         1738
              brown_spot    480     640          963
              brown_spot    640     480            2
              dead_heart    480     640         1442
            downy_mildew    480     640          620
                   hispa    480     640         1594
                  normal    480     640         1764
                  tungro    480     640         1088

3. Image aspect ratios:
aspect_ratio
0.7500    10403
1.3333        4

4. Image dimensions summary:
          width    height
count  10407.00 

## Phase 2.6: Resolution and Image-Property Distribution — Completed

The resolution audit identified two dimension combinations among 10,407 images.

* 480 × 640 pixels: 10,403 images (99.96%)
* 640 × 480 pixels: 4 images (0.04%)

The four landscape images belong to the bacterial leaf blight and brown spot classes, with two images in each class.

### Conclusion

The dataset has highly consistent image dimensions. No images were removed based on resolution or orientation. Original image files will be preserved, and resizing will be handled during model preprocessing rather than dataset auditing.


In [9]:
# ============================================================
# Phase 2.7 — Exact Duplicate Detection (SHA-256)
# ============================================================

import hashlib
import pandas as pd
from collections import defaultdict

image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

hash_to_paths = defaultdict(list)

all_image_paths = sorted(
    p for p in IMAGE_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in image_extensions
)

def sha256_file(file_path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()

    with open(file_path, "rb") as file:
        while True:
            chunk = file.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)

    return digest.hexdigest()

for index, image_path in enumerate(all_image_paths, start=1):
    file_hash = sha256_file(image_path)
    hash_to_paths[file_hash].append(image_path)

    if index % 1000 == 0:
        print(f"Hashed {index}/{len(all_image_paths)} images")

duplicate_groups = {
    file_hash: paths
    for file_hash, paths in hash_to_paths.items()
    if len(paths) > 1
}

duplicate_records = []

for group_number, (file_hash, paths) in enumerate(
    sorted(duplicate_groups.items()),
    start=1
):
    for image_path in paths:
        duplicate_records.append({
            "duplicate_group": group_number,
            "sha256": file_hash,
            "class_name": image_path.parent.name,
            "relative_path": str(image_path.relative_to(IMAGE_DIR)),
        })

duplicate_df = pd.DataFrame(duplicate_records)

if duplicate_df.empty:
    duplicate_df = pd.DataFrame(columns=[
        "duplicate_group",
        "sha256",
        "class_name",
        "relative_path",
    ])

duplicate_report = AUDIT_DIR / "rice_exact_duplicates.csv"
duplicate_df.to_csv(duplicate_report, index=False)

total_duplicate_groups = len(duplicate_groups)
images_in_groups = sum(
    len(paths) for paths in duplicate_groups.values()
)
redundant_copies = sum(
    len(paths) - 1 for paths in duplicate_groups.values()
)

print("\n" + "=" * 65)
print("RICE DATASET — EXACT DUPLICATE AUDIT")
print("=" * 65)
print("Total images checked:", len(all_image_paths))
print("Unique SHA-256 hashes:", len(hash_to_paths))
print("Exact duplicate groups:", total_duplicate_groups)
print("Images involved in duplicate groups:", images_in_groups)
print("Redundant copies beyond one per group:", redundant_copies)
print("Report saved to:", duplicate_report)

if not duplicate_df.empty:
    print("\nDuplicate groups by class:")
    print(
        duplicate_df.groupby("class_name")
        .size()
        .sort_values(ascending=False)
        .to_string()
    )

Hashed 1000/10407 images
Hashed 2000/10407 images
Hashed 3000/10407 images
Hashed 4000/10407 images
Hashed 5000/10407 images
Hashed 6000/10407 images
Hashed 7000/10407 images
Hashed 8000/10407 images
Hashed 9000/10407 images
Hashed 10000/10407 images

RICE DATASET — EXACT DUPLICATE AUDIT
Total images checked: 10407
Unique SHA-256 hashes: 10333
Exact duplicate groups: 72
Images involved in duplicate groups: 146
Redundant copies beyond one per group: 74
Report saved to: /home/zahidrashid/cate_model/Pearl_millet_Prediction/Rice_Dataset/Audit/rice_exact_duplicates.csv

Duplicate groups by class:
class_name
normal                      29
dead_heart                  25
brown_spot                  24
blast                       20
bacterial_leaf_blight       16
tungro                      16
hispa                       10
downy_mildew                 4
bacterial_panicle_blight     2


In [10]:
# ============================================================
# Phase 2.7.1 — Review Exact Duplicate Groups
# ============================================================

import pandas as pd

duplicate_report = AUDIT_DIR / "rice_exact_duplicates.csv"
duplicate_df = pd.read_csv(duplicate_report)

if duplicate_df.empty:
    print("No exact duplicate groups found.")
else:
    group_summary = (
        duplicate_df.groupby("duplicate_group")
        .agg(
            image_count=("relative_path", "count"),
            class_count=("class_name", "nunique"),
            classes=("class_name", lambda x: ", ".join(sorted(set(x)))),
            paths=("relative_path", lambda x: " | ".join(x)),
        )
        .reset_index()
    )

    group_summary["cross_class"] = group_summary["class_count"] > 1

    print("Total duplicate groups:", len(group_summary))
    print("Within-class groups:", (~group_summary["cross_class"]).sum())
    print("Cross-class groups:", group_summary["cross_class"].sum())

    print("\nDuplicate group summary:")
    display(group_summary)

    group_summary.to_csv(
        AUDIT_DIR / "rice_exact_duplicate_group_summary.csv",
        index=False
    )

    cross_class_groups = group_summary[group_summary["cross_class"]]

    print("\nCross-class duplicate groups — priority for manual review:")
    if cross_class_groups.empty:
        print("None found.")
    else:
        display(cross_class_groups)

Total duplicate groups: 72
Within-class groups: 72
Cross-class groups: 0

Duplicate group summary:


,duplicate_group,image_count,class_count,classes,paths,cross_class
0,1,2,1,dead_heart,dead_heart/104957.jpg | dead_heart/107011.jpg,False
1,2,2,1,brown_spot,brown_spot/100939.jpg | brown_spot/102827.jpg,False
2,3,3,1,dead_heart,dead_heart/101868.jpg | dead_heart/106481.jpg ...,False
3,4,2,1,normal,normal/100344.jpg | normal/107897.jpg,False
4,5,2,1,tungro,tungro/104116.jpg | tungro/106818.jpg,False
...,...,...,...,...,...,...
67,68,2,1,hispa,hispa/108645.jpg | hispa/109935.jpg,False
68,69,2,1,bacterial_leaf_blight,bacterial_leaf_blight/103344.jpg | bacterial_l...,False
69,70,2,1,normal,normal/106308.jpg | normal/107152.jpg,False
70,71,2,1,blast,blast/105038.jpg | blast/109773.jpg,False



Cross-class duplicate groups — priority for manual review:
None found.


In [11]:
# ============================================================
# Phase 2.7.2 — Create Exact-Duplicate Contact Sheets
# ============================================================

from PIL import Image, ImageDraw
from pathlib import Path
import pandas as pd
import math

duplicate_df = pd.read_csv(AUDIT_DIR / "rice_exact_duplicates.csv")

CONTACT_DIR = AUDIT_DIR / "exact_duplicate_contact_sheets"
CONTACT_DIR.mkdir(parents=True, exist_ok=True)

THUMB_W = 180
THUMB_H = 220
LABEL_H = 48
COLS = 4
GROUPS_PER_SHEET = 12

group_ids = sorted(duplicate_df["duplicate_group"].unique())

def create_duplicate_sheet(group_ids_subset, sheet_number):
    items = []

    for group_id in group_ids_subset:
        group_rows = duplicate_df[
            duplicate_df["duplicate_group"] == group_id
        ]

        for _, row in group_rows.iterrows():
            image_path = IMAGE_DIR / row["relative_path"]

            with Image.open(image_path) as img:
                thumb = img.convert("RGB")
                thumb.thumbnail((THUMB_W - 10, THUMB_H - 10))

            items.append({
                "group": group_id,
                "class": row["class_name"],
                "filename": Path(row["relative_path"]).name,
                "image": thumb.copy(),
            })

    rows = math.ceil(len(items) / COLS)
    cell_h = THUMB_H + LABEL_H

    sheet = Image.new(
        "RGB",
        (COLS * THUMB_W, rows * cell_h),
        "white"
    )
    draw = ImageDraw.Draw(sheet)

    for i, item in enumerate(items):
        x = (i % COLS) * THUMB_W
        y = (i // COLS) * cell_h

        img = item["image"]
        px = x + (THUMB_W - img.width) // 2
        py = y + 5

        sheet.paste(img, (px, py))

        label = (
            f"Group {item['group']} | {item['class']}\n"
            f"{item['filename']}"
        )
        draw.text((x + 5, y + THUMB_H), label, fill="black")

    output_path = CONTACT_DIR / f"exact_duplicates_{sheet_number:02d}.jpg"
    sheet.save(output_path, quality=92)
    return output_path

sheet_paths = []

for start in range(0, len(group_ids), GROUPS_PER_SHEET):
    subset = group_ids[start:start + GROUPS_PER_SHEET]
    sheet_number = start // GROUPS_PER_SHEET + 1
    output_path = create_duplicate_sheet(subset, sheet_number)
    sheet_paths.append(output_path)

print("Duplicate groups:", len(group_ids))
print("Contact sheets created:", len(sheet_paths))
print("Saved under:", CONTACT_DIR)

for path in sheet_paths:
    print(path)

Duplicate groups: 72
Contact sheets created: 6
Saved under: /home/zahidrashid/cate_model/Pearl_millet_Prediction/Rice_Dataset/Audit/exact_duplicate_contact_sheets
/home/zahidrashid/cate_model/Pearl_millet_Prediction/Rice_Dataset/Audit/exact_duplicate_contact_sheets/exact_duplicates_01.jpg
/home/zahidrashid/cate_model/Pearl_millet_Prediction/Rice_Dataset/Audit/exact_duplicate_contact_sheets/exact_duplicates_02.jpg
/home/zahidrashid/cate_model/Pearl_millet_Prediction/Rice_Dataset/Audit/exact_duplicate_contact_sheets/exact_duplicates_03.jpg
/home/zahidrashid/cate_model/Pearl_millet_Prediction/Rice_Dataset/Audit/exact_duplicate_contact_sheets/exact_duplicates_04.jpg
/home/zahidrashid/cate_model/Pearl_millet_Prediction/Rice_Dataset/Audit/exact_duplicate_contact_sheets/exact_duplicates_05.jpg
/home/zahidrashid/cate_model/Pearl_millet_Prediction/Rice_Dataset/Audit/exact_duplicate_contact_sheets/exact_duplicates_06.jpg


## Phase 2.7: Exact Duplicate Detection — Completed

### Method

SHA-256 hashes were calculated for all 10,407 extracted images. Files with identical hashes were grouped as exact duplicates.

### Results

* Exact duplicate groups: 72
* Images involved in duplicate groups: 146
* Redundant copies beyond one representative per group: 74
* Cross-class duplicate groups: 0

All duplicate groups were inspected using generated contact sheets. The visual review found no unexpected issues.

### Decision

The original image files will be preserved. A separate exclusion manifest will identify one retained representative per duplicate group and the redundant copies to exclude from the deduplicated working dataset.

Exact duplicate detection does not identify all visually similar images; perceptual near-duplicate analysis will be conducted separately.


In [12]:
# ============================================================
# Phase 2.7.3 — Create Exact-Duplicate Exclusion Manifest
# ============================================================

import pandas as pd
from pathlib import Path

duplicate_df = pd.read_csv(AUDIT_DIR / "rice_exact_duplicates.csv")

manifest_records = []

if not duplicate_df.empty:
    for group_id, group in duplicate_df.groupby("duplicate_group"):
        group = group.sort_values("relative_path")
        representative = group.iloc[0]["relative_path"]

        for _, row in group.iterrows():
            is_representative = row["relative_path"] == representative

            manifest_records.append({
                "duplicate_group": int(group_id),
                "class_name": row["class_name"],
                "relative_path": row["relative_path"],
                "sha256": row["sha256"],
                "decision": (
                    "retain_representative"
                    if is_representative
                    else "exclude_exact_duplicate"
                ),
                "representative_path": representative,
                "visual_review": "confirmed_consistent",
            })

manifest_df = pd.DataFrame(manifest_records)

if manifest_df.empty:
    manifest_df = pd.DataFrame(columns=[
        "duplicate_group",
        "class_name",
        "relative_path",
        "sha256",
        "decision",
        "representative_path",
        "visual_review",
    ])

manifest_path = AUDIT_DIR / "rice_exact_duplicate_exclusion_manifest.csv"
manifest_df.to_csv(manifest_path, index=False)

print("=" * 65)
print("EXACT-DUPLICATE EXCLUSION MANIFEST")
print("=" * 65)
print("Rows in manifest:", len(manifest_df))
print(
    "Representatives to retain:",
    (manifest_df["decision"] == "retain_representative").sum()
)
print(
    "Redundant copies marked for exclusion:",
    (manifest_df["decision"] == "exclude_exact_duplicate").sum()
)
print("Original images deleted: 0")
print("Manifest saved to:", manifest_path)

print("\nDecision counts:")
print(manifest_df["decision"].value_counts().to_string())

EXACT-DUPLICATE EXCLUSION MANIFEST
Rows in manifest: 146
Representatives to retain: 72
Redundant copies marked for exclusion: 74
Original images deleted: 0
Manifest saved to: /home/zahidrashid/cate_model/Pearl_millet_Prediction/Rice_Dataset/Audit/rice_exact_duplicate_exclusion_manifest.csv

Decision counts:
decision
exclude_exact_duplicate    74
retain_representative      72


## Phase 2.8: Perceptual Near-Duplicate Detection

### Objective

Identify visually similar images whose file contents differ and therefore were not detected by SHA-256 exact hashing.

### Method

Perceptual hashes will be calculated for the original extracted images. Candidate pairs will be identified using Hamming distance between hashes.

### Important limitation

A low perceptual-hash distance indicates visual similarity, not necessarily duplicate physical specimens. Similar symptoms, leaf shapes, and backgrounds can occur naturally across different samples.

All candidate pairs will be reviewed before any exclusion decision. Original images will remain unchanged.


In [13]:
# ============================================================
# Phase 2.8 — Step 1: Check Perceptual Hash Dependencies
# ============================================================

try:
    import imagehash
    from PIL import Image

    print("Pillow: available")
    print("ImageHash: available")
    print("ImageHash version:", getattr(imagehash, "__version__", "unknown"))

except ImportError as exc:
    print("Required library is missing:", exc)
    print("Run this in a notebook cell: %pip install ImageHash")

Pillow: available
ImageHash: available
ImageHash version: 4.3.2


In [14]:
# ============================================================
# Phase 2.8 — Step 2: Calculate Perceptual Hashes
# ============================================================

import imagehash
import pandas as pd
from PIL import Image
from pathlib import Path

image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

all_image_paths = sorted(
    p for p in IMAGE_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in image_extensions
)

phash_records = []
hash_errors = []

for index, image_path in enumerate(all_image_paths, start=1):
    try:
        with Image.open(image_path) as img:
            # Normalize orientation before hashing, without modifying files
            img = img.convert("RGB")
            phash = imagehash.phash(img)

        phash_records.append({
            "relative_path": str(image_path.relative_to(IMAGE_DIR)),
            "class_name": image_path.parent.name,
            "phash": str(phash),
        })

    except Exception as exc:
        hash_errors.append({
            "relative_path": str(image_path.relative_to(IMAGE_DIR)),
            "error": str(exc),
        })

    if index % 1000 == 0:
        print(f"Processed {index}/{len(all_image_paths)} images")

phash_df = pd.DataFrame(phash_records)

phash_path = AUDIT_DIR / "rice_perceptual_hashes.csv"
phash_df.to_csv(phash_path, index=False)

pd.DataFrame(hash_errors).to_csv(
    AUDIT_DIR / "rice_perceptual_hash_errors.csv",
    index=False
)

print("\n" + "=" * 65)
print("PERCEPTUAL HASH GENERATION")
print("=" * 65)
print("Images found:", len(all_image_paths))
print("Hashes generated:", len(phash_records))
print("Hashing errors:", len(hash_errors))
print("Hash file:", phash_path)

Processed 1000/10407 images
Processed 2000/10407 images
Processed 3000/10407 images
Processed 4000/10407 images
Processed 5000/10407 images
Processed 6000/10407 images
Processed 7000/10407 images
Processed 8000/10407 images
Processed 9000/10407 images
Processed 10000/10407 images

PERCEPTUAL HASH GENERATION
Images found: 10407
Hashes generated: 10407
Hashing errors: 0
Hash file: /home/zahidrashid/cate_model/Pearl_millet_Prediction/Rice_Dataset/Audit/rice_perceptual_hashes.csv


In [15]:
# ============================================================
# Phase 2.8 — Step 3: Find Candidate Near-Duplicate Pairs
# ============================================================

from itertools import combinations
from collections import Counter

phash_df = pd.read_csv(AUDIT_DIR / "rice_perceptual_hashes.csv")

hash_entries = [
    (
        row["relative_path"],
        row["class_name"],
        imagehash.hex_to_hash(row["phash"])
    )
    for _, row in phash_df.iterrows()
]

thresholds = [2, 4, 6]
pair_records = []

# Compare images only once per pair
for i in range(len(hash_entries)):
    path_a, class_a, hash_a = hash_entries[i]

    for j in range(i + 1, len(hash_entries)):
        path_b, class_b, hash_b = hash_entries[j]

        distance = hash_a - hash_b

        if distance <= max(thresholds):
            pair_records.append({
                "path_a": path_a,
                "class_a": class_a,
                "path_b": path_b,
                "class_b": class_b,
                "hamming_distance": distance,
                "cross_class": class_a != class_b,
            })

pairs_df = pd.DataFrame(pair_records)

if pairs_df.empty:
    pairs_df = pd.DataFrame(columns=[
        "path_a", "class_a", "path_b", "class_b",
        "hamming_distance", "cross_class"
    ])

pairs_path = AUDIT_DIR / "rice_near_duplicate_candidates.csv"
pairs_df.to_csv(pairs_path, index=False)

print("=" * 65)
print("NEAR-DUPLICATE CANDIDATE SUMMARY")
print("=" * 65)

for threshold in thresholds:
    selected = pairs_df[
        pairs_df["hamming_distance"] <= threshold
    ]

    print(f"\nDistance <= {threshold}:")
    print("Candidate pairs:", len(selected))
    print("Cross-class pairs:", int(selected["cross_class"].sum()))

print("\nPairs saved to:", pairs_path)

NEAR-DUPLICATE CANDIDATE SUMMARY

Distance <= 2:
Candidate pairs: 1051
Cross-class pairs: 27

Distance <= 4:
Candidate pairs: 2961
Cross-class pairs: 99

Distance <= 6:
Candidate pairs: 5705
Cross-class pairs: 176

Pairs saved to: /home/zahidrashid/cate_model/Pearl_millet_Prediction/Rice_Dataset/Audit/rice_near_duplicate_candidates.csv


In [16]:

# Phase 2.8 — Step 4: Inspect Exact-Duplicate Audit Structure

import pandas as pd

exact_path = AUDIT_DIR / "rice_exact_duplicate_exclusion_manifest.csv"
pairs_path = AUDIT_DIR / "rice_near_duplicate_candidates.csv"

exact_df = pd.read_csv(exact_path)
pairs_df = pd.read_csv(pairs_path)

print("EXACT-DUPLICATE MANIFEST")
print("Rows:", len(exact_df))
print("Columns:", exact_df.columns.tolist())
display(exact_df.head(10))

print("\nNEAR-DUPLICATE CANDIDATES")
print("Rows:", len(pairs_df))
print("Columns:", pairs_df.columns.tolist())

print("\nPairs with perceptual distance 0:")
zero_pairs = pairs_df[pairs_df["hamming_distance"] == 0]
print("Count:", len(zero_pairs))
display(zero_pairs.head(10))

print("\nCandidate distance distribution:")
print(pairs_df["hamming_distance"].value_counts().sort_index())

EXACT-DUPLICATE MANIFEST
Rows: 146
Columns: ['duplicate_group', 'class_name', 'relative_path', 'sha256', 'decision', 'representative_path', 'visual_review']


,duplicate_group,class_name,relative_path,sha256,decision,representative_path,visual_review
0,1,dead_heart,dead_heart/104957.jpg,00a0421aac5dc257ec484290b3de911d35388a7c126778...,retain_representative,dead_heart/104957.jpg,confirmed_consistent
1,1,dead_heart,dead_heart/107011.jpg,00a0421aac5dc257ec484290b3de911d35388a7c126778...,exclude_exact_duplicate,dead_heart/104957.jpg,confirmed_consistent
2,2,brown_spot,brown_spot/100939.jpg,01521e52e3ff77d458f947f40573aa34b49413d93b786c...,retain_representative,brown_spot/100939.jpg,confirmed_consistent
3,2,brown_spot,brown_spot/102827.jpg,01521e52e3ff77d458f947f40573aa34b49413d93b786c...,exclude_exact_duplicate,brown_spot/100939.jpg,confirmed_consistent
4,3,dead_heart,dead_heart/101868.jpg,04a15f1350b961d459aeec1e5d661c9b5453fa0781cacd...,retain_representative,dead_heart/101868.jpg,confirmed_consistent
5,3,dead_heart,dead_heart/106481.jpg,04a15f1350b961d459aeec1e5d661c9b5453fa0781cacd...,exclude_exact_duplicate,dead_heart/101868.jpg,confirmed_consistent
6,3,dead_heart,dead_heart/110000.jpg,04a15f1350b961d459aeec1e5d661c9b5453fa0781cacd...,exclude_exact_duplicate,dead_heart/101868.jpg,confirmed_consistent
7,4,normal,normal/100344.jpg,058136dc4c2d1873d19816c14e78dbd16b94be92c7b992...,retain_representative,normal/100344.jpg,confirmed_consistent
8,4,normal,normal/107897.jpg,058136dc4c2d1873d19816c14e78dbd16b94be92c7b992...,exclude_exact_duplicate,normal/100344.jpg,confirmed_consistent
9,5,tungro,tungro/104116.jpg,0793d3c8d1b3f6933ee9f9d4aebe5ae03a35fd3fc8a981...,retain_representative,tungro/104116.jpg,confirmed_consistent



NEAR-DUPLICATE CANDIDATES
Rows: 5705
Columns: ['path_a', 'class_a', 'path_b', 'class_b', 'hamming_distance', 'cross_class']

Pairs with perceptual distance 0:
Count: 229


,path_a,class_a,path_b,class_b,hamming_distance,cross_class
0,bacterial_leaf_blight/100133.jpg,bacterial_leaf_blight,bacterial_leaf_blight/103032.jpg,bacterial_leaf_blight,0,False
2,bacterial_leaf_blight/100169.jpg,bacterial_leaf_blight,bacterial_leaf_blight/100513.jpg,bacterial_leaf_blight,0,False
16,bacterial_leaf_blight/100635.jpg,bacterial_leaf_blight,bacterial_leaf_blight/107812.jpg,bacterial_leaf_blight,0,False
21,bacterial_leaf_blight/100998.jpg,bacterial_leaf_blight,bacterial_leaf_blight/106984.jpg,bacterial_leaf_blight,0,False
26,bacterial_leaf_blight/101173.jpg,bacterial_leaf_blight,bacterial_leaf_blight/108353.jpg,bacterial_leaf_blight,0,False
58,bacterial_leaf_blight/102038.jpg,bacterial_leaf_blight,bacterial_leaf_blight/108125.jpg,bacterial_leaf_blight,0,False
66,bacterial_leaf_blight/102353.jpg,bacterial_leaf_blight,bacterial_leaf_blight/104800.jpg,bacterial_leaf_blight,0,False
81,bacterial_leaf_blight/102935.jpg,bacterial_leaf_blight,bacterial_leaf_blight/106698.jpg,bacterial_leaf_blight,0,False
87,bacterial_leaf_blight/102940.jpg,bacterial_leaf_blight,bacterial_leaf_blight/105014.jpg,bacterial_leaf_blight,0,False
95,bacterial_leaf_blight/103230.jpg,bacterial_leaf_blight,bacterial_leaf_blight/104776.jpg,bacterial_leaf_blight,0,False



Candidate distance distribution:
hamming_distance
0     229
2     822
4    1910
6    2744
Name: count, dtype: int64


In [17]:
print("Manifest columns:")
print(exact_df.columns.tolist())

print("\nManifest sample:")
display(exact_df.head(5))

print("\nExact-hash candidate sample:")
display(zero_pairs.head(5))

Manifest columns:
['duplicate_group', 'class_name', 'relative_path', 'sha256', 'decision', 'representative_path', 'visual_review']

Manifest sample:


,duplicate_group,class_name,relative_path,sha256,decision,representative_path,visual_review
0,1,dead_heart,dead_heart/104957.jpg,00a0421aac5dc257ec484290b3de911d35388a7c126778...,retain_representative,dead_heart/104957.jpg,confirmed_consistent
1,1,dead_heart,dead_heart/107011.jpg,00a0421aac5dc257ec484290b3de911d35388a7c126778...,exclude_exact_duplicate,dead_heart/104957.jpg,confirmed_consistent
2,2,brown_spot,brown_spot/100939.jpg,01521e52e3ff77d458f947f40573aa34b49413d93b786c...,retain_representative,brown_spot/100939.jpg,confirmed_consistent
3,2,brown_spot,brown_spot/102827.jpg,01521e52e3ff77d458f947f40573aa34b49413d93b786c...,exclude_exact_duplicate,brown_spot/100939.jpg,confirmed_consistent
4,3,dead_heart,dead_heart/101868.jpg,04a15f1350b961d459aeec1e5d661c9b5453fa0781cacd...,retain_representative,dead_heart/101868.jpg,confirmed_consistent



Exact-hash candidate sample:


,path_a,class_a,path_b,class_b,hamming_distance,cross_class
0,bacterial_leaf_blight/100133.jpg,bacterial_leaf_blight,bacterial_leaf_blight/103032.jpg,bacterial_leaf_blight,0,False
2,bacterial_leaf_blight/100169.jpg,bacterial_leaf_blight,bacterial_leaf_blight/100513.jpg,bacterial_leaf_blight,0,False
16,bacterial_leaf_blight/100635.jpg,bacterial_leaf_blight,bacterial_leaf_blight/107812.jpg,bacterial_leaf_blight,0,False
21,bacterial_leaf_blight/100998.jpg,bacterial_leaf_blight,bacterial_leaf_blight/106984.jpg,bacterial_leaf_blight,0,False
26,bacterial_leaf_blight/101173.jpg,bacterial_leaf_blight,bacterial_leaf_blight/108353.jpg,bacterial_leaf_blight,0,False


In [18]:

# Phase 2.8 — Step 4: Classify Exact vs. Near-Duplicate Pairs

import pandas as pd

exact_df = pd.read_csv(
    AUDIT_DIR / "rice_exact_duplicate_exclusion_manifest.csv"
)
pairs_df = pd.read_csv(
    AUDIT_DIR / "rice_near_duplicate_candidates.csv"
)

# Build a path-to-SHA256 lookup from the existing audit manifest.
sha_by_path = (
    exact_df[["relative_path", "sha256"]]
    .drop_duplicates(subset=["relative_path"])
    .set_index("relative_path")["sha256"]
    .to_dict()
)

pairs_df["sha256_a"] = pairs_df["path_a"].map(sha_by_path)
pairs_df["sha256_b"] = pairs_df["path_b"].map(sha_by_path)

pairs_df["byte_identical"] = (
    pairs_df["sha256_a"].notna()
    & pairs_df["sha256_b"].notna()
    & (pairs_df["sha256_a"] == pairs_df["sha256_b"])
)

print("Total candidate pairs:", len(pairs_df))
print("Byte-identical pairs:", int(pairs_df["byte_identical"].sum()))
print("Non-identical pairs:", int((~pairs_df["byte_identical"]).sum()))

print("\nSummary by perceptual distance:")
summary = (
    pairs_df.groupby("hamming_distance")
    .agg(
        candidate_pairs=("hamming_distance", "size"),
        byte_identical_pairs=("byte_identical", "sum"),
        cross_class_pairs=("cross_class", "sum"),
    )
    .reset_index()
)
display(summary)

# Save a new report without changing the original candidate file.
classified_path = (
    AUDIT_DIR / "rice_near_duplicate_candidates_classified.csv"
)
pairs_df.to_csv(classified_path, index=False)

print("\nClassified report saved to:", classified_path)

Total candidate pairs: 5705
Byte-identical pairs: 76
Non-identical pairs: 5629

Summary by perceptual distance:


,hamming_distance,candidate_pairs,byte_identical_pairs,cross_class_pairs
0,0,229,76,2
1,2,822,0,25
2,4,1910,0,72
3,6,2744,0,77



Classified report saved to: /home/zahidrashid/cate_model/Pearl_millet_Prediction/Rice_Dataset/Audit/rice_near_duplicate_candidates_classified.csv


In [19]:

# Phase 2.8 — Step 5: Verify Cross-Class Exact-Match Candidates

cross_class_exact = pairs_df[
    pairs_df["cross_class"] & pairs_df["byte_identical"]
].copy()

print("Cross-class pairs marked byte-identical:", len(cross_class_exact))

for _, row in cross_class_exact.iterrows():
    print("\nPair:")
    print("A:", row["path_a"], "| class:", row["class_a"])
    print("B:", row["path_b"], "| class:", row["class_b"])
    print("pHash distance:", row["hamming_distance"])
    print("SHA-256 A:", row["sha256_a"])
    print("SHA-256 B:", row["sha256_b"])
    print("Hashes equal:", row["sha256_a"] == row["sha256_b"])

Cross-class pairs marked byte-identical: 0


In [29]:

# Phase 2.8 — Step 6: Create a Contact Sheet for Cross-Class Pairs (distance <= 2)

from PIL import Image, ImageDraw
import pandas as pd
import math

pairs_df = pd.read_csv(
    AUDIT_DIR / "rice_near_duplicate_candidates_classified.csv"
)

review_pairs = pairs_df[
    (pairs_df["cross_class"] == True)
    & (pairs_df["hamming_distance"] <= 2)
    & (~pairs_df["byte_identical"])
].copy()

print("Cross-class pairs to review:", len(review_pairs))
print("Distance distribution:")
print(review_pairs["hamming_distance"].value_counts().sort_index())

sheet_dir = AUDIT_DIR / "near_duplicate_review"
sheet_dir.mkdir(parents=True, exist_ok=True)

thumb_w, thumb_h = 220, 190
label_h = 58
pair_gap = 18
margin = 16
pairs_per_sheet = 12

def make_pair_sheet(batch, sheet_number):
    cell_w = thumb_w * 2 + pair_gap
    cell_h = thumb_h + label_h
    cols = 2
    rows = math.ceil(len(batch) / cols)

    canvas = Image.new(
        "RGB",
        (margin * 2 + cols * cell_w, margin * 2 + rows * cell_h),
        "white"
    )
    draw = ImageDraw.Draw(canvas)

    for idx, (_, pair) in enumerate(batch.iterrows()):
        cell_x = margin + (idx % cols) * cell_w
        cell_y = margin + (idx // cols) * cell_h

        for side, path_col, class_col in [
            ("A", "path_a", "class_a"),
            ("B", "path_b", "class_b"),
        ]:
            rel_path = pair[path_col]
            image_path = IMAGE_DIR / rel_path
            x = cell_x if side == "A" else cell_x + thumb_w + pair_gap

            try:
                with Image.open(image_path) as im:
                    im = im.convert("RGB")
                    im.thumbnail((thumb_w, thumb_h))
                    canvas.paste(im, (x, cell_y))
            except Exception as exc:
                draw.text((x, cell_y), f"Could not load: {exc}", fill="red")

            draw.text(
                (x, cell_y + thumb_h + 3),
                f"{side}: {pair[class_col]}",
                fill="black"
            )
            draw.text(
                (x, cell_y + thumb_h + 20),
                rel_path[:34],
                fill="black"
            )

        draw.text(
            (cell_x, cell_y + thumb_h + 39),
            f"Pair {pair.name} | pHash distance: {pair['hamming_distance']}",
            fill="blue"
        )

    out_path = sheet_dir / f"cross_class_near_duplicates_{sheet_number:02d}.jpg"
    canvas.save(out_path, quality=92)
    return out_path

review_pairs = review_pairs.reset_index(drop=True)
sheet_paths = []

for start in range(0, len(review_pairs), pairs_per_sheet):
    batch = review_pairs.iloc[start:start + pairs_per_sheet]
    sheet_paths.append(make_pair_sheet(batch, len(sheet_paths) + 1))

print("\nContact sheets created:")
for path in sheet_paths:
    print(path)


Cross-class pairs to review: 27
Distance distribution:
hamming_distance
0     2
2    25
Name: count, dtype: int64

Contact sheets created:
/home/zahidrashid/cate_model/Pearl_millet_Prediction/Rice_Dataset/Audit/near_duplicate_review/cross_class_near_duplicates_01.jpg
/home/zahidrashid/cate_model/Pearl_millet_Prediction/Rice_Dataset/Audit/near_duplicate_review/cross_class_near_duplicates_02.jpg
/home/zahidrashid/cate_model/Pearl_millet_Prediction/Rice_Dataset/Audit/near_duplicate_review/cross_class_near_duplicates_03.jpg


In [30]:

# Phase 2.8 — Step 7: Record Cross-Class Visual Review

import pandas as pd

pairs_path = AUDIT_DIR / "rice_near_duplicate_candidates_classified.csv"
pairs_df = pd.read_csv(pairs_path)

review_pairs = pairs_df[
    (pairs_df["cross_class"] == True)
    & (pairs_df["hamming_distance"] <= 2)
    & (~pairs_df["byte_identical"])
].copy()

review_pairs = review_pairs.reset_index(drop=True)
review_pairs.insert(0, "review_pair_id", range(1, len(review_pairs) + 1))

review_pairs["visual_review_decision"] = "confirmed_duplicate_by_user"
review_pairs["review_note"] = (
    "User confirmed duplicate after reviewing cross-class contact sheets; "
    "verify source image and labels before exclusion."
)
review_pairs["recommended_action"] = "investigate_cross_class_label_conflict"

review_path = AUDIT_DIR / "rice_cross_class_near_duplicate_review.csv"
review_pairs.to_csv(review_path, index=False)

print("Pairs recorded:", len(review_pairs))
print("Distinct class combinations:")
display(
    review_pairs.groupby(["class_a", "class_b"])
    .size()
    .reset_index(name="pair_count")
    .sort_values("pair_count", ascending=False)
)

print("\nReview manifest saved to:", review_path)
print("\nOriginal images were not modified.")


Pairs recorded: 27
Distinct class combinations:


,class_a,class_b,pair_count
1,blast,downy_mildew,10
5,hispa,normal,5
0,bacterial_leaf_streak,brown_spot,4
2,blast,hispa,3
4,downy_mildew,tungro,2
6,hispa,tungro,2
3,brown_spot,downy_mildew,1



Review manifest saved to: /home/zahidrashid/cate_model/Pearl_millet_Prediction/Rice_Dataset/Audit/rice_cross_class_near_duplicate_review.csv

Original images were not modified.


In [31]:

# Phase 2.8 — Step 8: Prepare Same-Class Near-Duplicate Review

import pandas as pd

pairs_df = pd.read_csv(
    AUDIT_DIR / "rice_near_duplicate_candidates_classified.csv"
)

same_class_candidates = pairs_df[
    (~pairs_df["cross_class"])
    & (~pairs_df["byte_identical"])
    & (pairs_df["hamming_distance"] <= 2)
].copy()

same_class_candidates = same_class_candidates.reset_index(drop=True)
same_class_candidates.insert(
    0, "review_pair_id", range(1, len(same_class_candidates) + 1)
)

shortlist_path = AUDIT_DIR / "rice_same_class_near_duplicate_shortlist.csv"
same_class_candidates.to_csv(shortlist_path, index=False)

print("Same-class non-identical candidate pairs (distance <= 2):",
      len(same_class_candidates))

print("\nPairs by class:")
display(
    same_class_candidates.groupby("class_a")
    .size()
    .reset_index(name="candidate_pairs")
    .sort_values("candidate_pairs", ascending=False)
)

print("\nDistance distribution:")
print(same_class_candidates["hamming_distance"].value_counts().sort_index())

print("\nShortlist saved to:", shortlist_path)
print("Original images were not modified.")


Same-class non-identical candidate pairs (distance <= 2): 948

Pairs by class:


,class_a,candidate_pairs
7,hispa,213
3,blast,141
5,dead_heart,115
4,brown_spot,96
8,normal,96
1,bacterial_leaf_streak,89
9,tungro,88
6,downy_mildew,82
0,bacterial_leaf_blight,19
2,bacterial_panicle_blight,9



Distance distribution:
hamming_distance
0    151
2    797
Name: count, dtype: int64

Shortlist saved to: /home/zahidrashid/cate_model/Pearl_millet_Prediction/Rice_Dataset/Audit/rice_same_class_near_duplicate_shortlist.csv
Original images were not modified.


In [32]:

# Phase 2.8 — Step 9: First Same-Class Review Batch

import pandas as pd
from PIL import Image, ImageDraw
import math

shortlist_path = AUDIT_DIR / "rice_same_class_near_duplicate_shortlist.csv"
shortlist = pd.read_csv(shortlist_path)

batch_size = 24
batch = shortlist.iloc[:batch_size].copy()

review_dir = AUDIT_DIR / "near_duplicate_review"
review_dir.mkdir(parents=True, exist_ok=True)

# Save the exact index of pairs shown on this sheet.
batch_index_path = review_dir / "same_class_batch_01_index.csv"
batch.to_csv(batch_index_path, index=False)

thumb_w, thumb_h = 200, 160
label_h = 48
pair_gap = 12
margin = 14
cols = 2
cell_w = thumb_w * 2 + pair_gap
cell_h = thumb_h + label_h
rows = math.ceil(len(batch) / cols)

canvas = Image.new(
    "RGB",
    (margin * 2 + cols * cell_w, margin * 2 + rows * cell_h),
    "white"
)
draw = ImageDraw.Draw(canvas)

for idx, (_, pair) in enumerate(batch.iterrows()):
    x0 = margin + (idx % cols) * cell_w
    y0 = margin + (idx // cols) * cell_h

    for side, path_col in [("A", "path_a"), ("B", "path_b")]:
        rel_path = pair[path_col]
        image_path = IMAGE_DIR / rel_path
        x = x0 if side == "A" else x0 + thumb_w + pair_gap

        try:
            with Image.open(image_path) as im:
                im = im.convert("RGB")
                im.thumbnail((thumb_w, thumb_h))
                canvas.paste(im, (x, y0))
        except Exception as exc:
            draw.text((x, y0), f"Load error: {exc}", fill="red")

        draw.text(
            (x, y0 + thumb_h + 2),
            f"{side}: {rel_path[:29]}",
            fill="black"
        )

    draw.text(
        (x0, y0 + thumb_h + 20),
        f"Pair ID {pair['review_pair_id']} | distance {pair['hamming_distance']}",
        fill="blue"
    )

sheet_path = review_dir / "same_class_near_duplicates_batch_01.jpg"
canvas.save(sheet_path, quality=92)

print("Pairs on sheet:", len(batch))
print("Contact sheet:", sheet_path)
print("Pair index:", batch_index_path)
print("\nClass distribution in this batch:")
display(batch["class_a"].value_counts().rename_axis("class_name").reset_index(name="pairs"))


Pairs on sheet: 24
Contact sheet: /home/zahidrashid/cate_model/Pearl_millet_Prediction/Rice_Dataset/Audit/near_duplicate_review/same_class_near_duplicates_batch_01.jpg
Pair index: /home/zahidrashid/cate_model/Pearl_millet_Prediction/Rice_Dataset/Audit/near_duplicate_review/same_class_batch_01_index.csv

Class distribution in this batch:


,class_name,pairs
0,bacterial_leaf_blight,19
1,bacterial_leaf_streak,5


In [33]:

# Phase 2.8 — Step 10: Record Same-Class Batch 01 Decisions

import pandas as pd

batch_index_path = (
    AUDIT_DIR / "near_duplicate_review" / "same_class_batch_01_index.csv"
)
batch = pd.read_csv(batch_index_path)

# Record your visual-review decision for every pair in this batch.
batch["visual_review_decision"] = "same_physical_image"
batch["review_status"] = "confirmed_by_user"
batch["review_note"] = (
    "User visually confirmed this pair as the same image/specimen."
)
batch["recommended_action"] = "exclude_one_copy_in_future_working_set"

review_path = (
    AUDIT_DIR / "near_duplicate_review"
    / "same_class_batch_01_reviewed.csv"
)
batch.to_csv(review_path, index=False)

print("Pairs reviewed:", len(batch))
print("\nDecisions:")
print(batch["visual_review_decision"].value_counts())

print("\nPairs by class:")
display(
    batch.groupby("class_a")
    .size()
    .reset_index(name="confirmed_duplicate_pairs")
    .sort_values("confirmed_duplicate_pairs", ascending=False)
)

print("\nReview manifest saved to:", review_path)
print("Original images deleted:", False)


Pairs reviewed: 24

Decisions:
visual_review_decision
same_physical_image    24
Name: count, dtype: int64

Pairs by class:


,class_a,confirmed_duplicate_pairs
0,bacterial_leaf_blight,19
1,bacterial_leaf_streak,5



Review manifest saved to: /home/zahidrashid/cate_model/Pearl_millet_Prediction/Rice_Dataset/Audit/near_duplicate_review/same_class_batch_01_reviewed.csv
Original images deleted: False
